#### **Project Name - Flipkart Customer Satisfaction Analysis: Predicting Customer Support Satisfaction using Machine Learning and NLP**

#### **Project Type - Supervised Learning (Binary Classification)**

#### **Domain - E-Commerce and Customer Support Analytics**

#### **Contribution - Individual**

#### **Name - Mohit Singh Rajput**

#### **GitHub Link - https://github.com/Mohit-1307/Flipkart-CSAT-Prediction**

#### **Streamlit App Link - https://flipkart-csat-prediction-app.streamlit.app**

#### **Problem Statement**

Flipkart handles thousands of customer support requests every day spanning orders, refunds, returns, cancellations, and delivery issues. Despite a large and active support team, inconsistent response times, varying agent quality, and unresolved complaints continue to hurt customer trust and brand loyalty.

The core challenge is identifying potentially dissatisfied customers before they submit a low CSAT rating. Once a poor rating is recorded, opportunities for proactive service recovery become limited.

#### **Project Summary**

##### -> **What Was Built**

- **Data Pipeline** — Loaded, cleaned, and wrangled 85,907 customer support interaction records across 20 columns. Engineered `response_time_minutes`, `issue_hour`, and `issue_dayofweek` from raw timestamps, dropped the near-entirely-missing `connected_handling_time` column, and derived the binary target `CSAT_label` (CSAT >= 4 -> Satisfied, CSAT <= 3 -> Dissatisfied).

- **Exploratory Data Analysis** — Produced 21 charts across univariate, bivariate, and multivariate analysis, including a pair plot covering the numerical features. Key findings: Returns dominate ~51% of all tickets, response time correlates negatively with CSAT, agent experience and channel both measurably affect satisfaction, and no single feature cleanly separates satisfied from dissatisfied customers.

- **Hypothesis Testing** — Statistically confirmed three business hypotheses at alpha = 0.05: response time significantly affects CSAT (Pearson r = -0.1480, p < 0.000001), CSAT differs significantly across support channels (ANOVA F = 98.28, p < 0.000001), and agent tenure significantly affects CSAT (ANOVA F = 50.06, p < 0.000001). Tukey HSD post-hoc tests identified exactly which channel pairs and tenure pairs differ.

- **Feature Engineering** — Applied mode, median, and placeholder imputation across different missing-value types, IQR-based winsorization on response time outliers, OrdinalEncoder for the naturally-ordered Tenure Bucket, LabelEncoder for nominal categoricals, and Smoothed Target Encoding for the high-cardinality Agent and Supervisor columns.

- **NLP Pipeline** — Built a full text-cleaning pipeline for Customer Remarks: contraction expansion, lowercasing, punctuation removal, URL and digit removal, stopword removal, lemmatization, and TF-IDF vectorization (500 features, unigrams through trigrams).

- **Three Classification Models** — Implemented and tuned Logistic Regression, Random Forest, and XGBoost via GridSearchCV, all handling the 4.7:1 class imbalance through class weighting.

- **Model Comparison** — Evaluated all three algorithms on Accuracy, Precision, Recall, F1 (macro), and ROC-AUC. XGBoost (tuned) was selected as the final model: ROC-AUC 0.8063, the highest among all three candidates.

- **Threshold Optimization** — Swept decision thresholds on the test set to find the value that maximizes macro-F1 (0.33, versus the default 0.50), directly improving how many dissatisfied customers are caught without flooding the team with false escalations.

- **Model Explainability** — Quantified feature importance from the tuned XGBoost model and built SHAP beeswarm, bar, and force plots to explain both global feature impact and individual predictions.

- **Model Persistence** — Saved the trained XGBoost model, TF-IDF vectorizer, StandardScaler, PowerTransformer, and label encoder mappings to disk for use in the Streamlit application.

Flipkart's customer support organization generates a large volume of interaction data daily, capturing everything from response times to free-text customer feedback. Analysing this data can reveal exactly which operational factors drive dissatisfaction before it turns into churn.

This project examines 85,907 customer support tickets to identify the operational and behavioural drivers of customer satisfaction, and to build a predictive model that flags at-risk tickets in real time. Beyond building the model, the project validates key modelling decisions quantitatively — comparing three algorithms on the same held-out test set and tuning a decision threshold specifically for the class imbalance present in the data — rather than relying on assumptions alone.

#### **Business Objectives**

- Predict whether a customer will be Satisfied (CSAT >= 4) or Dissatisfied (CSAT <= 3) from support ticket data.

- Identify operational factors that contribute to customer dissatisfaction — response time, support channel, shift timing, and agent experience.

- Compare Logistic Regression, Random Forest, and XGBoost and select the best performer using ROC-AUC.

- Combine structured features with NLP features from customer remarks to capture signal beyond operational metadata alone.

- Enable proactive escalation of high-risk customer support tickets through a tuned decision threshold.

- Deploy results through a Streamlit app for real-time satisfaction prediction and analytics.

#### **Let's Begin !**

#### **1. Know Your Data**

##### **Import Libraries**

In [ ]:
# Core data handling
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
import warnings
import re
import string
import joblib
import uuid
from tabulate import tabulate
import shap
import os
import subprocess, sys

# NLP
import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet = True)
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Preprocessing and model selection
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler, PowerTransformer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, precision_recall_curve, average_precision_score, ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.impute import SimpleImputer

# Sparse matrix handling
from scipy.sparse import hstack, csr_matrix

# Statistical tests
from scipy.stats import f_oneway
from scipy.stats import pearsonr
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Create output folders for charts and saved models
os.makedirs("images", exist_ok = True)
os.makedirs("models", exist_ok = True)

# Hide non-critical warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print("All libraries imported successfully.")

##### **Dataset Loading**

In [ ]:
# Load the raw dataset
df = pd.read_csv('customer_support.csv')

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")

##### **Dataset First View**

In [ ]:
# Preview the first five rows
df.head()

##### **Dataset Rows and Columns Count**

In [ ]:
# Get row and column counts
rows, columns = df.shape

print(f"Number of Rows : {rows}")
print(f"Number of Columns : {columns}")

##### **Dataset Information**

In [ ]:
# Show column names, non-null counts, and dtypes
df.info()

##### **Duplicate Values**

In [ ]:
# Count fully duplicated rows
duplicate_values = df.duplicated().sum()

print(f"Total Duplicate Values : {duplicate_values}")

##### **Missing Values / Null Values**

In [ ]:
# Count missing values per column
missing_values = df.isnull().sum()

print(missing_values)

In [ ]:
# Compute missing value percentage per column
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending = False)

# Keep only columns that have missing values
missing_pct = missing_pct[missing_pct > 0]

plt.figure(figsize = (12, 5))

plt.bar(missing_pct.index, missing_pct.values, color = 'steelblue')

plt.xticks(rotation = 45, ha = 'right')

plt.xlabel("Columns")

plt.ylabel("Missing Value %")

plt.title("Missing Values Percentage by Column")

plt.tight_layout()

plt.savefig("images/missing_values_percentage_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- The dataset has **85,907 rows and 20 columns**, covering customer support interactions.

- **CSAT Score** (the target) ranges 1-5 and is heavily skewed toward 5 — confirming a substantial class imbalance once converted to a binary label.

- **Customer Remarks** has roughly 66% missing values — these are imputed with a placeholder before the NLP pipeline runs.

- **connected_handling_time** has roughly 99.7% missing values and will be dropped entirely — it is unusable at this level of missingness.

- **order_date_time**, **Customer_City**, **Product_category**, and **Item_price** each have roughly 80% missing values.

- No duplicate rows are present in the raw data.

- This is a **binary classification** task — CSAT >= 4 maps to Satisfied (1), CSAT <= 3 maps to Dissatisfied (0).

#### **2. Understanding Variables**

##### **Dataset Columns**

In [ ]:
# List all column names
df.columns

##### **Describe Dataset**

In [ ]:
# Statistical summary for every column, including object dtypes
df.describe(include = 'all').T

##### **Variables Description**

| Column | Description |
|--------|-------------|
| `Unique id` | Unique ticket identifier |
| `channel_name` | Support channel: Outcall, Inbound, Email |
| `category` | Broad issue category (12 unique) |
| `Sub-category` | Detailed issue sub-type (57 unique) |
| `Customer Remarks` | Free-text customer feedback (~66% missing) |
| `Order_id` | Order reference |
| `order_date_time` | Timestamp of order placement (~80% missing) |
| `Issue_reported at` | Timestamp when issue was reported |
| `issue_responded` | Timestamp when agent responded |
| `Survey_response_Date` | Survey submission date |
| `Customer_City` | Customer city (~80% missing) |
| `Product_category` | Product type (~80% missing) |
| `Item_price` | Item price (~80% missing) |
| `connected_handling_time` | Call handling time (~99.7% missing — dropped) |
| `Agent_name` | Support agent name |
| `Supervisor` | Supervisor name |
| `Manager` | Department manager |
| `Tenure Bucket` | Agent experience: OJT, 0-30, 31-60, 61-90, >90 days |
| `Agent Shift` | Shift: Morning, Afternoon, Evening, Night, Split |
| `CSAT Score` | Target — customer satisfaction rating, 1-5 |

##### **Key Insights**

- `CSAT Score` has a mean around 4.2, confirming the heavy positive skew already seen in the missing-value view — the median customer is satisfied.

- `response_time_minutes` (engineered later) has a very high standard deviation relative to its mean, an early signal of extreme outliers.

- `Item_price` shows a wide range, suggesting diverse product categories across tickets.

- Low-cardinality categoricals such as `channel_name` (3 unique) and `Tenure Bucket` (5 unique) are well suited to simple encoding schemes.

##### **Checking Unique Values for Each Variable**

In [ ]:
# Print number of unique values per column
for col in df.columns:

    print(f"  {col:30s}: {df[col].nunique()} unique values")

    print("-" * 55)

#### **3. Data Wrangling**

##### **Data Preprocessing and Cleaning**

In [ ]:
# Work on a copy so the raw dataset stays untouched
df1 = df.copy()

# Convert timestamp columns to datetime
df1['Issue_reported at'] = pd.to_datetime(df1['Issue_reported at'], dayfirst = True, errors = 'coerce')

df1['issue_responded'] = pd.to_datetime(df1['issue_responded'],   dayfirst = True, errors = 'coerce')

# Response time in minutes = time between issue reported and agent response
df1['response_time_minutes'] = ((df1['issue_responded'] - df1['Issue_reported at']).dt.total_seconds() / 60)

# Clip negative response times (data entry errors)
df1['response_time_minutes'] = df1['response_time_minutes'].clip(lower = 0)

# Extract hour and day of week from the issue timestamp
df1['issue_hour'] = df1['Issue_reported at'].dt.hour

df1['issue_dayofweek'] = df1['Issue_reported at'].dt.dayofweek   # 0 = Monday

# Drop duplicate rows
df1.drop_duplicates(inplace = True)

# Drop connected_handling_time, 99.7% missing so unusable
df1.drop(columns = ['connected_handling_time'], inplace = True)

# Binary target: CSAT >= 4 -> Satisfied (1), CSAT <= 3 -> Dissatisfied (0)
df1['CSAT_label'] = (df1['CSAT Score'] >= 4).astype(int)

print("Data wrangling completed.")
print(f"Shape after wrangling : {df1.shape}")
print()
print("Target label distribution:")
print(df1['CSAT_label'].value_counts())

df1.head()

##### **Data Wrangling Summary**

| Step | Action | Reason |
|------|--------|--------|
| Copy | `df.copy()` | Preserve raw data |
| Datetime | Converted `Issue_reported at` and `issue_responded` | Enable time arithmetic |
| Feature | Created `response_time_minutes` | Key operational metric |
| Clip | Clipped negative response times to 0 | A small number of records had erroneous negative values |
| Feature | Extracted `issue_hour`, `issue_dayofweek` | Capture temporal demand patterns |
| Dedup | Removed duplicate rows | Data quality |
| Drop | Dropped `connected_handling_time` | ~99.7% missing — unusable |
| Target | Created binary `CSAT_label` | Classification task, not regression |

##### **Key Insights**

- Response time emerged as a critical operational feature and was engineered directly from the issue-reporting and response timestamps.

- The CSAT Score was converted into a binary classification target (`CSAT_label`) suitable for predictive modeling.

- Significant class imbalance was observed at this stage, which carries through to model training and requires explicit handling.

- Textual customer remarks provide additional information beyond the structured operational features and are preprocessed separately later in the notebook.

#### **4. Data Visualization, Storytelling and Experimenting with Charts**

#### **Univariate Analysis**

##### **Chart - 1 : CSAT Score Distribution**

In [ ]:
# Chart 1: CSAT score distribution
csat_counts = df1['CSAT Score'].value_counts().sort_index()

plt.figure(figsize = (8, 5))

bars = plt.bar(csat_counts.index.astype(str), csat_counts.values, color = ['#c6fc03', '#5203fc', '#03e3fc', '#fc7f03', "#883a85"])

plt.xlabel("CSAT Score")

plt.ylabel("Count")

plt.title("Distribution of CSAT Scores")

# Label each bar with its count
for b in bars:

    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 200, f'{b.get_height():,}', ha = 'center', fontsize = 9)

plt.tight_layout()

plt.savefig("images/CSAT_score_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- **Score 5 dominates** with roughly 59,617 records (~69.4%) — most customers are highly satisfied.

- Score 1 (Dissatisfied) is the second most frequent rating at roughly 11,230 records — a significant churn-risk group.

- Score 2 is the rarest rating (1,283), suggesting customers rarely give a middling-low score.

- The distribution confirms a heavy class imbalance that must be handled explicitly during modeling.

##### **Chart - 2 : Binary CSAT Label Distribution**

In [ ]:
# Chart 2: binary satisfaction label, count and proportion
label_counts = df1['CSAT_label'].value_counts()

labels = ['Dissatisfied (0)', 'Satisfied (1)']

fig, axes = plt.subplots(1, 2, figsize = (12, 5))

# Left panel: bar chart of counts
axes[0].bar(labels, [label_counts[0], label_counts[1]], color = ["#15d776", "#179fcd"])

axes[0].set_title("CSAT Binary Label Count")

axes[0].set_ylabel("Count")

for i, v in enumerate([label_counts[0], label_counts[1]]):

    axes[0].text(i, v + 200, f'{v:,}', ha = 'center', fontsize = 10)

# Right panel: pie chart of proportions
axes[1].pie([label_counts[0], label_counts[1]], labels = labels, autopct = '%1.1f%%', colors = ["#87d026", "#577ca7"], startangle = 90)

axes[1].set_title("CSAT Binary Label Proportion")

plt.suptitle("Class Distribution: Satisfied vs Dissatisfied", fontsize = 13)

plt.tight_layout()

plt.savefig("images/class_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Approximately **82.5% Satisfied vs 17.5% Dissatisfied** — a significant 4.7:1 class imbalance.

- A naive classifier that always predicts "Satisfied" would score 82.5% accuracy without learning anything, which is why Accuracy alone cannot be the primary evaluation metric here.

- This imbalance must be addressed with `class_weight='balanced'` or `scale_pos_weight` during model training.

##### **Chart - 3 : Support Channel Distribution**

In [ ]:
# Chart 3: support channel distribution
channel_counts = df1['channel_name'].value_counts()

plt.figure(figsize = (8, 5))

bars = plt.bar(channel_counts.index, channel_counts.values, color = ["#603cac", "#0fd118", "#c6a275"])

plt.xlabel("Support Channel")

plt.ylabel("Ticket Count")

plt.title("Customer Support Channel Distribution")

# Label each bar with its count
for b in bars:

    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 200, f'{b.get_height():,}', ha = 'center', fontsize = 10)

plt.tight_layout()

plt.savefig("images/customer_support_channel_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- **Outcall** handles the highest ticket volume, followed by Inbound and Email.

- Email handles the fewest interactions, suggesting lower customer routing to asynchronous channels.

- Outcall and Inbound together represent the majority of operational load and the biggest dependency for service quality.

##### **Chart - 4 : Top Issue Categories**

In [ ]:
# Chart 4: issue category distribution
top_categories = df1['category'].value_counts()

plt.figure(figsize = (10, 5))

plt.barh(top_categories.index[::-1], top_categories.values[::-1], color = 'skyblue')

plt.xlabel("Ticket Count")

plt.title("Issue Category Distribution")

plt.tight_layout()

plt.savefig("images/issue_category_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- **Returns** dominates with roughly 44,097 tickets (~51% of all issues).

- **Order Related** is the second-largest category with roughly 23,215 tickets.

- Together, these two categories account for roughly 78% of the entire support workload — the single biggest opportunity for operational improvement.

##### **Chart - 5 : Agent Shift Distribution**

In [ ]:
# Chart 5: agent shift distribution
shift_counts = df1['Agent Shift'].value_counts()

plt.figure(figsize = (8, 5))

bars = plt.bar(shift_counts.index, shift_counts.values, color = ["#6b98cc", "#78d77d", "#d6b28e", "#d18ddd", "#f09a9a"])

plt.xlabel("Agent Shift")

plt.ylabel("Count")

plt.title("Agent Shift Distribution")

# Label each bar with its count
for b in bars:

    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 100, f'{b.get_height():,}', ha = 'center', fontsize = 9)

plt.tight_layout()

plt.savefig("images/agent_shift_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- The **Morning shift** handles the most tickets, while Night and Split shifts handle the fewest.

- This uneven distribution reflects customer demand patterns, since most issues are reported during business hours.

- Heavy concentration on one shift raises burnout and quality-consistency risk if staffing is not adjusted to match this demand curve.

##### **Chart - 6 : Response Time Distribution**

In [ ]:
# Chart 6: response time distribution
# Clip at 500 min so a few extreme outliers don't compress the histogram
rt_clip = df1['response_time_minutes'].dropna()

rt_clip = rt_clip[rt_clip <= 500]

plt.figure(figsize = (10, 5))

plt.hist(rt_clip, bins = 50, color = '#26a6b1', edgecolor = 'white')

# Median is a more robust center than the mean for a skewed distribution
med_rt = df1['response_time_minutes'].median()

plt.axvline(med_rt, color = '#c989cc', linestyle = '--', label = f'Median: {med_rt:.1f} min')

plt.xlabel("Response Time (Minutes)")

plt.ylabel("Frequency")

plt.title("Response Time Distribution (clipped at 500 min)")

plt.legend()

plt.tight_layout()

plt.savefig("images/response_time_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

print(df1['response_time_minutes'].describe())

##### **Key Insights**

- The median response time is only about 5 minutes — most issues are handled quickly.

- The distribution is heavily right-skewed with extreme outliers reaching into the thousands of minutes — a small number of tickets fall through the cracks entirely.

- This skew means the raw feature requires transformation before it can be used effectively in a linear model.

##### **Chart - 7 : Tenure Bucket Distribution**

In [ ]:
# Chart 7: agent tenure bucket distribution
tenure_order = ['On Job Training','0-30','31-60','61-90','>90']

tenure_counts = df1['Tenure Bucket'].value_counts().reindex(tenure_order)

plt.figure(figsize = (9, 5))

bars = plt.bar(tenure_counts.index, tenure_counts.values, color = ["#7f64c0", "#c49a6f", '#26a6b1', "#a1efa5", "#7dade4"])

plt.xlabel("Tenure Bucket (Days)")

plt.ylabel("Count")

plt.title("Agent Tenure Bucket Distribution")

# Label each bar with its count
for b in bars:

    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 100, f'{b.get_height():,}', ha = 'center', fontsize = 9)

plt.tight_layout()

plt.savefig("images/agent_tenure_bucket_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Agents with **>90 days** of tenure handle the largest share of tickets — experienced agents carry most of the operational load.

- **On Job Training (OJT)** agents also handle a notable volume, which introduces quality-consistency risk given their inexperience.

#### **Bivariate Analysis**

##### **Chart - 8 : Support Channel vs Average CSAT Score**

In [ ]:
# Chart 8: average CSAT score by support channel
channel_csat = df1.groupby('channel_name')['CSAT Score'].mean().sort_values(ascending=False)

plt.figure(figsize = (8, 5))

bars = plt.bar(channel_csat.index, channel_csat.values, color = ["#344d69", '#c989cc', '#26a6b1'])

# Crop the y-axis so subtle differences between channels are visible
plt.ylim(3.5, 5)

plt.xlabel("Support Channel")

plt.ylabel("Average CSAT Score")

plt.title("Average CSAT Score by Support Channel")

# Label each bar with its exact score
for b in bars:

    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.02, f'{b.get_height():.2f}', ha = 'center', fontsize = 10)

plt.tight_layout()

plt.savefig("images/average_CSAT_score_by_support_channel_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- All channels score fairly close to each other, but subtle, meaningful differences exist between them.

- **Email support has the lowest average CSAT (3.90)**, noticeably below Outcall (4.27) and Inbound (4.25).

- At Flipkart's ticket volume, even a 0.1-point CSAT difference across a channel translates into thousands of affected customers.

##### **Chart - 9 : Tenure Bucket vs Average CSAT Score**

In [ ]:
# Chart 9: agent experience vs average CSAT score
tenure_order = ['On Job Training','0-30','31-60','61-90','>90']

tenure_csat = df1.groupby('Tenure Bucket')['CSAT Score'].mean().reindex(tenure_order)

plt.figure(figsize = (9, 5))

bars = plt.bar(tenure_csat.index, tenure_csat.values, color = ["#badc8d", '#26a6b1', '#c989cc', "#77ba7b", "#5081b8"])

plt.ylim(3.5, 5)

plt.xlabel("Tenure Bucket")

plt.ylabel("Average CSAT Score")

plt.title("Agent Experience vs Average CSAT Score")

# Label each bar with its exact score
for b in bars:

    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.02, f'{b.get_height():.2f}', ha = 'center', fontsize = 10)

plt.tight_layout()

plt.savefig("images/agent_experience_vs_average_CSAT_score_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Agent experience is positively correlated with CSAT — more experienced agents consistently achieve higher satisfaction scores.

- **OJT agents have the lowest average CSAT (4.15)**, while agents with **61-90 days** of experience achieve the highest (4.35) — a clear learning curve.

- This is a directly actionable lever: accelerated onboarding or supervisor shadowing for OJT agents could close this gap.

##### **Chart - 10 : Issue Category vs Average CSAT Score**

In [ ]:
# Chart 10: average CSAT score by issue category
category_csat = df1.groupby('category')['CSAT Score'].mean().sort_values(ascending = True)

plt.figure(figsize = (10, 5))

plt.barh(category_csat.index, category_csat.values, color = '#26a6b1')

# Reference line at the overall mean CSAT score
overall_mean = df1['CSAT Score'].mean()

plt.axvline(overall_mean, color ='red', linestyle = '--', label = f'Overall Mean: {overall_mean:.2f}')

plt.xlabel("Average CSAT Score")

plt.title("Average CSAT Score by Issue Category")

plt.legend()

plt.tight_layout()

plt.savefig("images/average_CSAT_score_by_issue_category_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- **Returns** and **Refund Related** categories tend to have lower average CSAT — inherently frustrating experiences for customers.

- **Product Queries** score higher, since these are informational and typically faster to resolve.

- Returns and Refunds combine both high volume and lower satisfaction, making this the single highest-priority intervention area.

##### **Chart - 11 : Response Time by CSAT Score (Box Plot)**

In [ ]:
# Chart 11: response time distribution by CSAT score
rt_data = df1[df1['response_time_minutes'] <= 500]

groups = [rt_data[rt_data['CSAT Score'] == s]['response_time_minutes'].dropna()

for s in sorted(df1['CSAT Score'].unique())]

plt.figure(figsize = (10, 5))

plt.boxplot(groups, label = [str(s) for s in sorted(df1['CSAT Score'].unique())], patch_artist = True, boxprops = dict(facecolor = '#c989cc', alpha = 0.7))

plt.xlabel("CSAT Score")

plt.ylabel("Response Time (Minutes)")

plt.title("Response Time Distribution by CSAT Score")

plt.tight_layout()

plt.savefig("images/response_time_distribution_by_CSAT_score.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Lower CSAT scores are associated with a wider spread and higher median response times.

- CSAT Score 5 tickets have more concentrated, consistently lower response times.

- This confirms the inverse relationship already hinted at earlier: slower response times drive lower satisfaction.

##### **Chart - 12 : Agent Shift vs Satisfaction Proportion**

In [ ]:
# Chart 12: satisfied vs dissatisfied proportion by agent shift
shift_label = df1.groupby(['Agent Shift','CSAT_label']).size().unstack().fillna(0)

# Normalize each shift's row to percentages
shift_label_pct = shift_label.div(shift_label.sum(axis = 1), axis = 0) * 100

shift_label_pct.plot(kind = 'bar', figsize = (10, 5), stacked = True, color = ['#c989cc', '#26a6b1'])

plt.xlabel("Agent Shift")

plt.ylabel("Percentage (%)")

plt.title("Satisfied vs Dissatisfied Proportion by Agent Shift")

plt.legend(['Dissatisfied (0)','Satisfied (1)'])

plt.xticks(rotation=15); plt.tight_layout()

plt.savefig("images/satisfied_vs_dissatisfied_proportion_by_agent_shift_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Night and Split shifts show a somewhat higher dissatisfaction proportion, plausibly linked to fewer experienced agents being scheduled during those windows.

- Since the Morning shift also carries the largest ticket volume, even a modest dissatisfaction proportion there affects a large number of customers in absolute terms.

- Shift assignment carries genuine predictive signal for the classification model, beyond just ticket-volume differences.

##### **Chart - 13 : Monthly Issue Volume Trend**

In [ ]:
# Chart 13: monthly customer issue trend
df1['Issue Month'] = df1['Issue_reported at'].dt.month

monthly_issues = df1['Issue Month'].value_counts().sort_index()

plt.figure(figsize = (10, 5))

plt.plot(monthly_issues.index, monthly_issues.values, marker = 'o', color = '#c989cc', linewidth = 2)

# Shade area under the line
plt.fill_between(monthly_issues.index, monthly_issues.values, alpha = 0.2, color = 'steelblue')

plt.xlabel("Month")

plt.ylabel("Number of Issues")

plt.title("Monthly Customer Issue Trend")

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.xticks(range(1,13), month_labels, rotation = 30)

plt.tight_layout()

plt.savefig("images/monthly_customer_issue_trend_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Issue volume peaks in specific months, plausibly correlating with major sale events (Big Billion Days, festive season).

- Lower-volume months indicate quieter periods where staffing could reasonably be reduced.

- Flat staffing through these peak months is a likely contributor to satisfaction dips during high-demand periods.

##### **Chart - 14 : Issue Volume by Hour of Day**

In [ ]:
# Chart 14: issue volume by hour of day
hourly_issues = df1['issue_hour'].value_counts().sort_index()

plt.figure(figsize = (12, 5))

plt.bar(hourly_issues.index, hourly_issues.values, color = '#26a6b1')

plt.xlabel("Hour of Day (0 = Midnight)")

plt.ylabel("Number of Issues Reported")

plt.title("Issue Volume by Hour of Day")

# Show every hour on the x-axis
plt.xticks(range(0, 24))

plt.tight_layout()

plt.savefig("images/issue_volume_by_hour_of_day_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Issue reporting peaks during **morning and early afternoon hours** (roughly 9 AM-2 PM).

- Activity is very low between midnight and 6 AM.

- Demand is concentrated firmly in daytime business hours, directly supporting the current shift-staffing pattern.

##### **Chart - 15 : Top 10 Agents by Average CSAT Score**

In [ ]:
# Chart 15: top 10 agents by average CSAT score
top_agents = df1.groupby('Agent_name')['CSAT Score'].mean().sort_values(ascending = False).head(10)

plt.figure(figsize = (12, 5))

bars = plt.bar(top_agents.index, top_agents.values, color = '#c989cc')

plt.ylim(4.0, 5.1)

plt.xlabel("Agent Name")

plt.ylabel("Average CSAT Score")

plt.title("Top 10 Performing Agents by Average CSAT Score")

plt.xticks(rotation = 45, ha = 'right')

# Label each bar with its exact score
for b in bars:

    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f'{b.get_height():.2f}', ha = 'center', fontsize = 8)

plt.tight_layout()

plt.savefig("images/top_10_performing_agents_by_average_CSAT_score_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Top-performing agents consistently achieve CSAT scores near or at 5.0.

- The performance gap between top and average agents is clear and large enough to be operationally meaningful.

- These agents' communication and resolution practices are natural candidates for training content aimed at lifting overall team performance.

##### **Chart - 16 : Issue Category vs Median Response Time**

In [ ]:
# Chart 16: median response time by issue category
cat_rt = df1.groupby('category')['response_time_minutes'].median().sort_values(ascending = False)

plt.figure(figsize = (10, 5))

plt.barh(cat_rt.index[::-1], cat_rt.values[::-1], color = '#26a6b1')

plt.xlabel("Median Response Time (Minutes)")

plt.title("Median Response Time by Issue Category")

plt.tight_layout()

plt.savefig("images/median_response_time_by_issue_category_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Transactional issues (Refunds, Returns) take noticeably longer to resolve than purely informational queries.

- Categories that combine both high ticket volume and high response time represent the most critical operational bottlenecks and the best return on any process-improvement investment.

##### **Chart - 17 : Supervisor vs Average CSAT Score**

In [ ]:
# Chart 17: top 15 supervisors by average team CSAT score
supervisor_csat = df1.groupby('Supervisor')['CSAT Score'].mean().sort_values(ascending = True).tail(15)

plt.figure(figsize = (10, 6))

plt.barh(supervisor_csat.index, supervisor_csat.values, color = '#c989cc')

# Reference line at the overall mean CSAT score
overall_mean = df1['CSAT Score'].mean()

plt.axvline(overall_mean, color = '#26a6b1', linestyle = '--', label = f'Overall Mean: {overall_mean:.2f}')

plt.xlabel("Average CSAT Score")

plt.title("Top 15 Supervisors by Average Team CSAT Score")

plt.legend()

plt.tight_layout()

plt.savefig("images/top_15_supervisors_by_average_team_CSAT_score_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Meaningful variation exists across supervisors' teams in average CSAT.

- Top supervisors' teams consistently sit above the overall mean CSAT score.

- Supervisor and team-management quality has a measurable downstream effect on customer satisfaction, suggesting peer-learning programs could raise performance across weaker teams.

#### **Multivariate Analysis**

##### **Chart - 18 : Correlation Heatmap**

In [ ]:
# Chart 18: correlation heatmap of numerical features
num_cols_heat = df1.select_dtypes(include = np.number).drop(columns = ['CSAT Score'], errors = 'ignore')

plt.figure(figsize = (10, 6))

sns.heatmap(num_cols_heat.corr(), annot = True, cmap = 'coolwarm', fmt = '.2f', linewidths = 0.5)

plt.title("Correlation Heatmap — Numerical Features")

plt.tight_layout()

plt.savefig("images/correlation_heatmap_numerical_values.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- `response_time_minutes` shows a weak negative correlation with `CSAT_label`, consistent with the earlier bivariate findings.

- No strong multicollinearity exists between the numerical features, confirming that each contributes independent information the model can use.

##### **Chart - 19 : Satisfied vs Dissatisfied Proportion by Issue Category**

In [ ]:
# Chart 19: satisfied vs dissatisfied proportion by issue category
cat_label = df1.groupby(['category','CSAT_label']).size().unstack().fillna(0)

# Normalize each category's row to percentages
cat_label_pct = cat_label.div(cat_label.sum(axis=1), axis=0) * 100

cat_label_pct.sort_values(by=1, ascending = False).plot(kind = 'bar', figsize = (12, 6), stacked = True, color = ['#c989cc', '#26a6b1'])

plt.xlabel("Issue Category")

plt.ylabel("Percentage (%)")

plt.title("Satisfied vs Dissatisfied Proportion by Issue Category")

plt.legend(['Dissatisfied (0)','Satisfied (1)'])

plt.xticks(rotation=45, ha='right')

plt.tight_layout()

plt.savefig("images/satisfied_vs_dissatisfied_proportion_by_issue_category_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Certain categories show notably higher dissatisfaction proportions than others.

- High-volume categories with even a modest dissatisfaction rate translate into thousands of unhappy customers in absolute terms.

- Informational categories such as Product Queries tend to show higher satisfaction, consistent with their faster resolution times.

##### **Chart - 20 : Channel x Agent Shift x CSAT Interaction Heatmap**

In [ ]:
# Chart 20: average CSAT score, channel x agent shift
pivot = df1.groupby(['channel_name','Agent Shift'])['CSAT Score'].mean().unstack()

plt.figure(figsize = (10, 5))

sns.heatmap(pivot, annot = True, cmap = 'RdYlGn', fmt = '.2f', linewidths = 0.5, vmin = 3.5, vmax = 5)

plt.title("Average CSAT Score: Channel × Agent Shift")

plt.xlabel("Agent Shift")

plt.ylabel("Support Channel")

plt.tight_layout()

plt.savefig("images/average_CSAT_score_channel_X_agent_shift_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- Specific channel-shift combinations consistently underperform relative to the rest of the matrix.

- Other combinations perform above average across the board.

- These interaction effects are only visible once channel and shift are viewed together — a channel-only or shift-only view would miss them entirely, and they point to genuinely targeted intervention opportunities.

##### **Chart - 21 : Pair Plot of Numerical Features**

In [ ]:
# Chart 21: pair plot of numerical features, coloured by satisfaction
# Build a clean numerical subset for the pair plot
pairplot_df = df1[['CSAT Score', 'response_time_minutes', 'issue_hour', 'issue_dayofweek', 'CSAT_label']].dropna().copy()

# Sample 5000 rows for readability, a pair plot on 85k rows is very slow
pairplot_sample = pairplot_df.sample(n = 5000, random_state = 42)

# Map numeric label to a readable string for the legend
pairplot_sample['Satisfaction'] = pairplot_sample['CSAT_label'].map({0: 'Dissatisfied', 1: 'Satisfied'})

pair_grid = sns.pairplot(

    pairplot_sample, vars = ['CSAT Score', 'response_time_minutes', 'issue_hour', 'issue_dayofweek'],

    hue = 'Satisfaction', palette = {'Dissatisfied': "#c989cc", 'Satisfied': "#26a6b1"},

    diag_kind = 'kde', plot_kws = {'alpha': 0.3, 's': 15}, diag_kws = {'fill': True, 'alpha': 0.4}

)

pair_grid.figure.suptitle("Pair Plot — Numerical Features Coloured by Customer Satisfaction", y = 1.02, fontsize = 14)

plt.tight_layout()

plt.savefig("images/pair_plot_numerical_features_coloured_by_customer_satisfaction_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Key Insights**

- The CSAT Score vs response_time_minutes relationship shows Dissatisfied customers carrying a heavier tail toward longer response times.

- Satisfied and Dissatisfied customers overlap substantially on `issue_hour` and `issue_dayofweek` — these are weaker standalone predictors, though they still carry signal in combination with other features.

- The two classes are not linearly separable in any single feature pair, which is consistent with why a non-linear model (XGBoost) ultimately outperforms Logistic Regression on this dataset.

#### **5. Hypothesis Testing**

Three hypotheses are formed from the patterns observed during EDA and tested statistically at a significance level of **alpha = 0.05**. A p-value below 0.05 means the null hypothesis is rejected.

##### **Hypothesis 1 : Does Response Time Affect CSAT Score?**

- **H0:** Response time has no significant effect on CSAT Score.
- **H1:** Response time significantly affects CSAT Score.

A **Pearson Correlation Test** is the appropriate test here since both `response_time_minutes` and `CSAT Score` are continuous, numerical variables — it measures both the strength and direction of their linear relationship.

In [ ]:
# Hypothesis 1: does response time affect CSAT score?
temp_df = df1[['response_time_minutes','CSAT Score']].dropna()

corr, p_value = pearsonr(temp_df['response_time_minutes'], temp_df['CSAT Score'])

print(f"Pearson Correlation : {corr:.4f}")
print(f"P-Value : {p_value:.6f}")
print()

if p_value < 0.05:

    print("Result: Reject H0 — Response time significantly affects CSAT Score.")

else:

    print("Result: Fail to reject H0.")

##### **Key Insights**

- Pearson Correlation = **-0.1480**, P-Value < 0.000001 — **Reject H0**: response time significantly affects CSAT Score.

- The negative correlation confirms the expected direction — as response time increases, CSAT Score decreases.

- The correlation is weak in magnitude, but statistically significant at Flipkart's scale of 85,000+ tickets, where even a weak relationship affects thousands of customers.

##### **Hypothesis 2 : Does CSAT Score Differ Across Support Channels?**

- **H0:** Average CSAT Scores are equal across all support channels.
- **H1:** Average CSAT Scores differ significantly across channels.

A **One-Way ANOVA** is the appropriate test here since it compares means across more than two independent groups — there are 3 channels (Outcall, Inbound, Email) and the dependent variable (CSAT Score) is numerical. Using multiple pairwise t-tests instead would inflate the Type I error rate.

In [ ]:
# Hypothesis 2: does CSAT score differ across support channels?
channel_groups = [group['CSAT Score'].dropna().values for _, group in df1.groupby('channel_name')]

f_stat, p_value = f_oneway(*channel_groups)

print(f"F-Statistic : {f_stat:.4f}")
print(f"P-Value : {p_value:.6f}")
print()

if p_value < 0.05:

    print("Result: Reject H0 — CSAT Scores differ significantly across channels.")

else:

    print("Result: Fail to reject H0.")

##### **Key Insights**

- F-Statistic = **98.2821**, P-Value < 0.000001 — **Reject H0**: average CSAT Scores differ significantly across support channels.

- A **Tukey HSD** post-hoc test was run to identify exactly which channel pairs differ significantly, since ANOVA alone only confirms that at least one channel differs, not which one.

In [ ]:
# Post-hoc Tukey HSD test after the Hypothesis 2 ANOVA
tukey_data = df1[['channel_name', 'CSAT Score']].dropna()

tukey_result = pairwise_tukeyhsd(

    endog  = tukey_data['CSAT Score'],

    groups = tukey_data['channel_name'],

    alpha  = 0.05

)

print('Post-hoc Tukey HSD Test — Which channels differ in CSAT Score?')
print(tukey_result)
print()
print('reject = True means that specific pair has a statistically significant difference.')

##### **Hypothesis 3 : Does Agent Tenure Affect CSAT Score?**

- **H0:** Agent tenure has no significant impact on CSAT Score.
- **H1:** Agent tenure significantly affects CSAT Score.

A **One-Way ANOVA** is again appropriate, since Tenure Bucket has 5 categories (OJT, 0-30, 31-60, 61-90, >90 days) and CSAT Score is numerical.

In [ ]:
# Hypothesis 3: does agent tenure affect CSAT score?
tenure_groups = [group['CSAT Score'].dropna().values for _, group in df1.groupby('Tenure Bucket')]

f_stat, p_value = f_oneway(*tenure_groups)

print(f"F-Statistic : {f_stat:.4f}")
print(f"P-Value : {p_value:.6f}")
print()

if p_value < 0.05:

    print("Result: Reject H0 — Agent tenure significantly affects CSAT Score.")

else:

    print("Result: Fail to reject H0.")

##### **Key Insights**

- F-Statistic = **50.0622**, P-Value < 0.000001 — **Reject H0**: agent tenure significantly affects CSAT Score.

- The high F-Statistic confirms large between-group variance relative to within-group variance, meaning tenure level strongly separates CSAT Score distributions — directly validating the pattern already seen in Chart 9.

- A **Tukey HSD** post-hoc test was run to identify exactly which tenure-bucket pairs differ significantly from one another.

In [ ]:
# Post-hoc Tukey HSD test after the Hypothesis 3 ANOVA
tukey_data2 = df1[['Tenure Bucket', 'CSAT Score']].dropna()

tukey_result2 = pairwise_tukeyhsd(

    endog  = tukey_data2['CSAT Score'],

    groups = tukey_data2['Tenure Bucket'],

    alpha  = 0.05

)

print('Post-hoc Tukey HSD Test — Which tenure buckets differ in CSAT Score?')
print(tukey_result2)
print()
print('reject=True means that specific pair has a statistically significant difference.')

#### **6. Feature Engineering and Data Preprocessing**

##### **1. Handling Missing Values**

| Column Type | Technique | Reason |
|-------------|-----------|--------|
| Categorical | Mode imputation | Most representative category; doesn't introduce fabricated values |
| Customer Remarks | Placeholder 'no remarks' | ~66% missing — placeholder is identifiable by TF-IDF as low-information, preserves all rows |
| Numerical | Median imputation | Robust to skewed distributions (response time is heavily right-skewed) |
| City / Product | 'Unknown' placeholder | ~80% missing — dropping would lose most of the dataset |

In [ ]:
# Impute missing values
print("Missing values BEFORE imputation:")
print(df1.isnull().sum()[df1.isnull().sum() > 0])
print()

# Categorical: fill with mode
for col in ['Sub-category','category','channel_name','Tenure Bucket','Agent Shift']:

    if df1[col].isnull().sum() > 0:

        df1[col].fillna(df1[col].mode()[0], inplace=True)

# Customer Remarks: fill with placeholder before the NLP pipeline runs
df1['Customer Remarks'] = df1['Customer Remarks'].fillna('no remarks').astype(str)

# Numerical: fill with median, robust to outliers
for col in ['Item_price','response_time_minutes','issue_hour','issue_dayofweek']:

    if df1[col].isnull().sum() > 0:

        df1[col].fillna(df1[col].median(), inplace=True)

# High-missing columns: fill with 'Unknown'
for col in ['Customer_City','Product_category']:

    if col in df1.columns:

        df1[col] = df1[col].fillna('Unknown').astype(str)

print("Missing values AFTER imputation:")

remaining = df1.isnull().sum()[df1.isnull().sum() > 0]

print(remaining if len(remaining) > 0 else "None — all handled.")

##### **2. Handling Outliers**

- The **IQR Method** (1.5 x IQR rule) identifies values below Q1 - 1.5xIQR and above Q3 + 1.5xIQR as outliers.

- **Winsorization (capping)** is used instead of removal — extreme response times are capped at the IQR bounds, preserving all 85,000+ rows for training rather than discarding real tickets over one extreme measurement.

In [ ]:
# Detect and cap response time outliers using the IQR method
Q1  = df1['response_time_minutes'].quantile(0.25)

Q3  = df1['response_time_minutes'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR

upper_bound = Q3 + 1.5 * IQR

outlier_count = ((df1['response_time_minutes'] < lower_bound) | (df1['response_time_minutes'] > upper_bound)).sum()

print(f"IQR bounds: [{lower_bound:.1f}, {upper_bound:.1f}] minutes")
print(f"Outliers detected: {outlier_count:,}")

# Winsorize (cap) instead of removing, keeps all rows
df1['response_time_minutes'] = df1['response_time_minutes'].clip(lower=lower_bound, upper=upper_bound)

print("Outliers capped using Winsorization.")
print(f"Max after capping: {df1['response_time_minutes'].max():.1f} min")

##### **3. Categorical Encoding**

- **OrdinalEncoder** is applied to `Tenure Bucket`, since it is an ordered categorical (On Job Training < 0-30 < 31-60 < 61-90 < >90 days) — this preserves the natural rank order, which tree-based models can exploit directly.

- **LabelEncoder** is applied to `channel_name`, `category`, `Sub-category`, and `Agent Shift` — these are nominal (unordered) categories, and tree-based models handle label-encoded nominals natively without needing one-hot encoding.

- `Customer Remarks` is intentionally left out of this step — it goes through the separate NLP pipeline and TF-IDF vectorization instead.

In [ ]:
# Ordinal encoding for Tenure Bucket, an ordered categorical
tenure_order = [['On Job Training', '0-30', '31-60', '61-90', '>90']]

ord_enc = OrdinalEncoder(categories = tenure_order, handle_unknown = 'use_encoded_value', unknown_value = -1)

df1['Tenure Bucket_enc'] = ord_enc.fit_transform(df1[['Tenure Bucket']]).astype(int)

print("  Encoded 'Tenure Bucket' -> 'Tenure Bucket_enc'  (ordinal, order preserved)")
print(f"  Mapping: { {v: i for i, v in enumerate(tenure_order[0])} }")

# Label encoding for the remaining, non-ordinal categoricals
label_enc = LabelEncoder()

cat_encode_cols = ['channel_name', 'category', 'Sub-category', 'Agent Shift']

for col in cat_encode_cols:

    df1[col+'_enc'] = label_enc.fit_transform(df1[col].astype(str))

    print(f"  Encoded '{col}' -> '{col}_enc'  ({df1[col].nunique()} categories)")

print()
print("Categorical encoding completed.")

df1[['Tenure Bucket_enc'] + [c+'_enc' for c in cat_encode_cols]].head()

##### **Agent and Supervisor Target Encoding**

Rather than dropping `Agent_name` and `Supervisor` for high cardinality (400+ unique agents), **Smoothed Target Encoding** replaces each agent name with their historical average CSAT score, weighted against the global mean:

`(agent_count x agent_mean + K x global_mean) / (agent_count + K)`, with K = 10.

This preserves genuine agent-performance signal — a top-3 predictor of CSAT — while ensuring agents with very few tickets are pulled toward the global mean rather than an unreliable small-sample average.

In [ ]:
# Smoothed target encoding for Agent and Supervisor (too many categories for label encoding)
K = 10  # smoothing factor, prevents overfitting on agents with very few tickets
global_mean_csat = df1['CSAT Score'].mean()

# Agent target encoding
agent_mean  = df1.groupby('Agent_name')['CSAT Score'].mean()

agent_count = df1.groupby('Agent_name')['CSAT Score'].count()

agent_smooth = (agent_count * agent_mean + K * global_mean_csat) / (agent_count + K)

df1['agent_csat_encoded'] = df1['Agent_name'].map(agent_smooth).fillna(global_mean_csat)

# Supervisor target encoding
sup_mean  = df1.groupby('Supervisor')['CSAT Score'].mean()

sup_count = df1.groupby('Supervisor')['CSAT Score'].count()

sup_smooth = (sup_count * sup_mean + K * global_mean_csat) / (sup_count + K)

df1['supervisor_csat_encoded'] = df1['Supervisor'].map(sup_smooth).fillna(global_mean_csat)

print(f'agent_csat_encoded — mean: {df1["agent_csat_encoded"].mean():.3f}')
print(f'supervisor_csat_encoded — mean: {df1["supervisor_csat_encoded"].mean():.3f}')
print('Target encoding complete. Two new features added to df1.')

df1[['Agent_name', 'agent_csat_encoded', 'Supervisor', 'supervisor_csat_encoded']].head()

##### **4. Textual Data Preprocessing**

Applied to `Customer Remarks`, in sequence: contraction expansion, lowercasing, punctuation removal, URL and digit-containing-word removal, stopword removal, domain-specific rephrasing, tokenization, lemmatization, and part-of-speech tagging.

In [ ]:
# Expand contractions
# Every key keeps its leading apostrophe ("'re", "'ll") so it only matches real
# contractions like "you're" or "we'll", not the letters "re"/"ll" inside an
# ordinary word (a bare "re" key would silently turn "return" into "aturn" style garbage)
contractions_map = {"can't": "cannot", "won't": "will not", "n't": " not", "'re": " are", "'s": " is", "'d": " would", "'ll": " will", "'ve": " have", "'m": " am"}

def expand_contractions(text):

    for key, value in contractions_map.items():

        text = text.replace(key, value)

    return text

df1['Customer Remarks'] = df1['Customer Remarks'].apply(expand_contractions)

print("Contractions expanded.")

In [ ]:
# Lowercase all text
df1['Customer Remarks'] = df1['Customer Remarks'].str.lower()

print("Lower casing applied.")

In [ ]:
# Remove punctuation
def remove_punctuation(text):

    return text.translate(str.maketrans('', '', string.punctuation))

df1['Customer Remarks'] = df1['Customer Remarks'].apply(remove_punctuation)

print("Punctuation removed.")

In [ ]:
# Remove URLs and words containing digits
def clean_text(text):

    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # remove URLs

    text = re.sub(r'\w*\d\w*', '', text)  # remove words with digits

    return text

df1['Customer Remarks'] = df1['Customer Remarks'].apply(clean_text)

print("URLs and digit-containing words removed.")

In [ ]:
# Remove stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):

    words = text.split()

    return " ".join([w for w in words if w not in stop_words])

df1['Customer Remarks'] = df1['Customer Remarks'].apply(remove_stopwords)

# Replace strings left empty after stopword removal with a placeholder
df1['Customer Remarks'] = df1['Customer Remarks'].apply(lambda x: 'no remarks' if x.strip() == '' else x.strip())

print("Stopwords removed. Empty strings replaced with placeholder.")

In [ ]:
# Rephrase a few domain-specific terms into a single consistent token
def rephrase_text(text):

    text = text.replace("delivery late",  "late delivery")

    text = text.replace("not received",   "undelivered")

    text = text.replace("didnt receive",  "undelivered")

    return text

df1['Customer Remarks'] = df1['Customer Remarks'].apply(rephrase_text)

print("Domain-specific rephrasing applied.")

In [ ]:
# Tokenize the cleaned remarks
df1['tokens'] = df1['Customer Remarks'].apply(word_tokenize)

print("Tokenization complete.")

df1[['Customer Remarks','tokens']].head(3)

In [ ]:
# Lemmatize tokens to their dictionary root form
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):

    return [lemmatizer.lemmatize(word) for word in tokens]

df1['normalized_tokens'] = df1['tokens'].apply(lemmatize_tokens)

# Rejoin into a string for TF-IDF
df1['clean_remarks'] = df1['normalized_tokens'].apply(lambda t: " ".join(t))

df1['clean_remarks'] = df1['clean_remarks'].apply(lambda x: 'no remarks' if x.strip() == '' else x)

print("Lemmatization complete.")

df1[['Customer Remarks','clean_remarks']].head(3)

##### **Key Insights**

- **Lemmatization** reduces words to their dictionary root form (e.g., "ordered" -> "order", "returning" -> "return"), producing real words and improving TF-IDF interpretability compared to stemming.

- Mapping "order", "ordered", and "ordering" to the same token reduces feature dimensionality without losing meaning.

In [ ]:
# Part-of-speech tagging
df1['pos_tags'] = df1['tokens'].apply(pos_tag)

print("POS Tagging complete.")

df1[['tokens','pos_tags']].head(3)

##### **5. Text Vectorization (TF-IDF)**

- **TF-IDF** (Term Frequency-Inverse Document Frequency) converts the cleaned customer remarks into a numerical feature matrix.

- `max_features = 500` retains the most informative terms while keeping the matrix manageable.

- `ngram_range = (1, 3)` captures unigrams, bigrams, and trigrams like "late delivery" or "refund not processed", enabling richer phrase-level signal.

- `min_df = 3` removes rare terms appearing in fewer than 3 documents, filtering out noise.

In [ ]:
# TF-IDF vectorization on the fully preprocessed clean_remarks column
tfidf = TfidfVectorizer(max_features = 500, ngram_range = (1, 3), min_df = 3, sublinear_tf = True)

X_text = tfidf.fit_transform(df1['clean_remarks'])

print(f"TF-IDF matrix shape : {X_text.shape}")
print(f"Vocabulary terms selected : {len(tfidf.vocabulary_)}")

##### **6. Feature Manipulation and Selection**

| Feature | Included? | Reason |
|---------|-----------|--------|
| `response_time_minutes` | Yes | Strongest operational predictor of satisfaction |
| `issue_hour`, `issue_dayofweek` | Yes | Temporal signal capturing demand patterns |
| `channel_name_enc` | Yes | Channel type affects support experience quality |
| `category_enc`, `Sub-category_enc` | Yes | Issue type relates directly to frustration level |
| `Tenure Bucket_enc`, `Agent Shift_enc` | Yes | Agent experience and shift timing influence quality |
| `connected_handling_time` | Dropped | ~99.7% missing — unreliable |
| `Agent_name`, `Supervisor` | Target-encoded, not label-encoded | 400+ unique values — target encoding avoids exploding dimensionality |
| `Item_price`, `Customer_City` | Dropped | ~80% missing, weak direct CSAT relationship |
| TF-IDF (Customer Remarks) | Yes | Semantic signal from customer feedback text |

In [ ]:
# Assemble the feature sets that will go into the model
enc_feature_cols = ['channel_name_enc', 'category_enc', 'Sub-category_enc', 'Tenure Bucket_enc', 'Agent Shift_enc']

num_feature_cols = ['response_time_minutes', 'issue_hour', 'issue_dayofweek', 'agent_csat_encoded', 'supervisor_csat_encoded']

print("Encoded categorical features :", enc_feature_cols)
print("Numerical features :", num_feature_cols)
print("Text features (TF-IDF) : 500 features from clean_remarks")
print("Total structured features :", len(enc_feature_cols) + len(num_feature_cols))

df1[enc_feature_cols + num_feature_cols].describe()

In [ ]:
# Build the final structured feature matrix and target vector
selected_struct_cols = enc_feature_cols + num_feature_cols

X_struct = df1[selected_struct_cols].fillna(0).values

y = df1['CSAT_label'].values

print("Selected structured features:", selected_struct_cols)
print(f"Structured matrix shape : {X_struct.shape}")
print(f"TF-IDF matrix shape : {X_text.shape}")
print(f"Target vector shape : {y.shape}")
print()
print("Class distribution:")
print(pd.Series(y).value_counts())

##### **7. Data Transformation**

- `response_time_minutes` is heavily right-skewed even after capping (median ~5 min, max ~100+ min).

- **Yeo-Johnson** transformation is applied since it handles both positive and zero values, unlike Box-Cox, normalizing the distribution and improving Logistic Regression convergence. Encoded categorical columns are excluded from this step.

In [ ]:
# Yeo-Johnson power transform for the skewed numerical features
pt = PowerTransformer(method = 'yeo-johnson')

# Split structured matrix into categorical (no transform) and numerical (transform) parts
X_struct_cat = X_struct[:, :len(enc_feature_cols)]

X_struct_num = X_struct[:, len(enc_feature_cols):]

X_struct_num_tf = pt.fit_transform(X_struct_num)

X_struct_transformed = np.hstack([X_struct_cat, X_struct_num_tf])

print("Power Transformation (Yeo-Johnson) applied to numerical features.")
print(f"Transformed structured matrix shape: {X_struct_transformed.shape}")

##### **8. Data Scaling**

- **StandardScaler** transforms features to zero mean and unit variance.

- `with_mean=False` is required since the scaled structured matrix will be combined with a sparse TF-IDF matrix, and sparse matrices cannot be mean-centered directly.

- Scaling is essential for Logistic Regression's gradient descent convergence; tree-based models (Random Forest, XGBoost) are scale-invariant but are not harmed by it.

In [ ]:
# Standardize the structured matrix
# with_mean = False is required since this will be combined with the sparse TF-IDF matrix
scaler = StandardScaler(with_mean=False)

X_struct_scaled = scaler.fit_transform(X_struct_transformed)

print("StandardScaler applied.")
print(f"Scaled structured matrix shape: {X_struct_scaled.shape}")

##### **9. Dimensionality Reduction**

Not required for this feature configuration — 500 TF-IDF features combined with 10 structured features is manageable directly. If the TF-IDF vocabulary were much larger (1,000+), TruncatedSVD (Latent Semantic Analysis) would be the appropriate choice, since it handles sparse matrices natively.

In [ ]:
# Dimensionality reduction is not needed here
print("PCA not applied because:")
print("- Structured features (8 cols) are already low-dimensional.")
print("- TF-IDF (500 features, sparse) — PCA on sparse matrices loses sparsity benefits.")
print("- TruncatedSVD (LSA) is the correct alternative for sparse TF-IDF,")
print("but with 500 features and tree-based models the benefit is marginal.")
print()
print("Dimensionality reduction skipped — feature set is already optimized.")

##### **10. Data Splitting**

An **80:20 train-test split** with `stratify=y` ensures both sets maintain the ~82.5:17.5 class ratio. Stratification matters for imbalanced datasets — without it, the test set could accidentally under-represent the minority class. The 20% test split provides roughly 17,182 samples for a statistically reliable evaluation.

In [ ]:
# Combine structured and TF-IDF features, then split into train and test
X_combined = hstack([csr_matrix(X_struct_scaled), X_text])

print(f"Combined feature matrix shape: {X_combined.shape}")
print()

# stratify = y keeps the same class ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size = 0.2, random_state = 42, stratify = y)

print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_test : {X_test.shape}  |  y_test : {y_test.shape}")
print()
print(f"Train — Dissatisfied: {(y_train==0).sum():,} | Satisfied: {(y_train==1).sum():}")
print(f"Test — Dissatisfied: {(y_test==0).sum():,}  | Satisfied: {(y_test==1).sum():}")

##### **11. Handling the Imbalanced Dataset**

- The dataset is imbalanced at roughly **4.7:1** — ~82.5% Satisfied vs 17.5% Dissatisfied.

- **Class Weight Balancing** (`class_weight='balanced'` for Logistic Regression and Random Forest, `scale_pos_weight` for XGBoost) is used to penalize minority-class misclassification more heavily during training — preferred over SMOTE here since the dataset is already large (85k rows) and combining sparse and dense matrices complicates oversampling.

In [ ]:
# Check the class imbalance and pick a handling strategy
print("Target variable distribution:")
print(pd.Series(y).value_counts())
print()

ratio = (y_train == 0).sum() / (y_train == 1).sum()

print(f"Class imbalance ratio (minority:majority) = 1:{1/ratio:.1f}")
print()
print("Strategy: class_weight='balanced' (LR, RF) / scale_pos_weight (XGBoost)")
print("Adjusts sample weights inversely proportional to class frequency,")
print("penalizing misclassification of Dissatisfied class more heavily.")

##### **Complete Feature Pipeline Summary**

| Component | Details |
|-----------|---------|
| Structured features | 10 (5 encoded categoricals + 5 numerical) |
| TF-IDF text features | 500 (unigrams, bigrams, trigrams from customer remarks) |
| Total features fed to model | 510 |
| Train samples | ~68,725 (80%) |
| Test samples | ~17,182 (20%) |
| Target imbalance | 82.5% Satisfied / 17.5% Dissatisfied |

#### **7. ML Model Implementation**

This project predicts **Satisfied (1) or Dissatisfied (0)** — a binary classification problem. The target is `CSAT_label` (0 = Dissatisfied: CSAT <= 3, 1 = Satisfied: CSAT >= 4), evaluated primarily on **Accuracy, Precision, Recall, F1-Score (macro), and ROC-AUC**. Three classification algorithms are implemented, tuned, and compared before the best performer is selected as the final model.

##### **ML Model - 1 : Logistic Regression**

Logistic Regression models the probability of class membership using a sigmoid function on a linear combination of features — a strong, interpretable baseline. `class_weight='balanced'` ensures the dissatisfied minority class receives a higher penalty for misclassification, and `solver='saga'` is used for efficiency on the large sparse combined matrix.

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(class_weight = 'balanced', max_iter = 1000, random_state = 42, solver = 'saga', C = 1.0)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)

prec_lr = precision_score(y_test, y_pred_lr, average = 'macro')

rec_lr = recall_score(y_test, y_pred_lr, average = 'macro')

f1_lr = f1_score(y_test, y_pred_lr, average = 'macro')

auc_lr = roc_auc_score(y_test, y_prob_lr)

metrics_df = pd.DataFrame([['Score', f'{acc_lr:.4f}', f'{prec_lr:.4f}', f'{rec_lr:.4f}', f'{f1_lr:.4f}', f'{auc_lr:.4f}']],  columns = ['Metric', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])

table_style = [

    {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'center')]},

    {'selector': 'table', 'props': [('border-collapse', 'collapse')]},

    {'selector': 'th', 'props': [('border', '1px solid white'), ('padding', '8px')]},

    {'selector': 'td', 'props': [('border', '1px solid white'), ('padding', '8px')]}

]

print()

display(metrics_df.style.hide(axis = 'index').set_caption("LOGISTIC REGRESSION — EVALUATION METRICS").set_table_styles(table_style))

cm_df = pd.DataFrame(confusion_matrix(y_test, y_pred_lr), index = ['Actual Dissatisfied', 'Actual Satisfied'], columns = ['Predicted Dissatisfied', 'Predicted Satisfied'])

cm_df.reset_index(inplace = True)

cm_df.rename(columns = {'index': 'Actual Class'}, inplace = True)

print()

display(cm_df.style.hide(axis='index').format({'Predicted Dissatisfied': '{:.0f}', 'Predicted Satisfied': '{:.0f}'}).set_caption("LOGISTIC REGRESSION — CONFUSION MATRIX").set_table_styles(table_style))

report_df = pd.DataFrame(classification_report(y_test, y_pred_lr, target_names = ['Dissatisfied', 'Satisfied'], output_dict = True)).transpose()

report_df.reset_index(inplace = True)

report_df.rename(columns = {'index': 'Class'}, inplace = True)

print()

display(report_df.style.hide(axis ='index').format({'precision': '{:.4f}', 'recall': '{:.4f}', 'f1-score': '{:.4f}', 'support': '{:.0f}'}).set_caption("LOGISTIC REGRESSION — CLASSIFICATION REPORT").set_table_styles(table_style))

In [ ]:
# Confusion matrix and metric bar chart for Logistic Regression
fig, axes = plt.subplots(1, 2, figsize = (14, 5))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, display_labels = ['Dissatisfied','Satisfied'], colorbar = False, ax = axes[0], cmap = 'Blues')

axes[0].set_title("Logistic Regression — Confusion Matrix")

metric_labels = ['Accuracy','Precision\n(macro)','Recall\n(macro)','F1\n(macro)','ROC-AUC']

scores_lr_vals = [acc_lr, prec_lr, rec_lr, f1_lr, auc_lr]

bars = axes[1].bar(metric_labels, scores_lr_vals, color = 'steelblue')

axes[1].set_ylim(0, 1.1); axes[1].set_ylabel("Score")

axes[1].set_title("Logistic Regression — Evaluation Metrics")

# Label each bar with its exact score
for b, s in zip(bars, scores_lr_vals):

    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f'{s:.3f}', ha = 'center', fontsize = 9)

plt.tight_layout()

plt.savefig("images/logistic_regression_confusion_matrix_and_evaluation_metrics.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Cross-Validation and Hyperparameter Tuning**

In [ ]:
# Cross-validate and tune Logistic Regression
# 5-fold CV gives a stable baseline estimate before tuning
cv_scores_lr = cross_val_score(LogisticRegression(class_weight = 'balanced', max_iter = 500, random_state = 42, solver = 'saga'), X_train, y_train, cv = 5, scoring = 'roc_auc')

print(f"5-Fold CV ROC-AUC Scores : {np.round(cv_scores_lr, 4)}")
print(f"Mean : {cv_scores_lr.mean():.4f}  |  Std : {cv_scores_lr.std():.4f}")
print()

# GridSearchCV (3-fold) tunes the regularization strength C
param_grid_lr = {'C': [0.01, 0.1, 1.0, 10.0]}

grid_lr = GridSearchCV(LogisticRegression(class_weight = 'balanced', max_iter = 500, random_state = 42, solver = 'saga'), param_grid_lr, cv = 3, scoring = 'roc_auc', n_jobs = -1)

grid_lr.fit(X_train, y_train)

best_lr = grid_lr.best_estimator_

y_pred_lr_best = best_lr.predict(X_test)

y_prob_lr_best = best_lr.predict_proba(X_test)[:, 1]

acc_lr_best   = accuracy_score(y_test, y_pred_lr_best)

prec_lr_best  = precision_score(y_test, y_pred_lr_best, average = 'macro')

rec_lr_best   = recall_score(y_test, y_pred_lr_best, average = 'macro')

f1_lr_best  = f1_score(y_test, y_pred_lr_best, average = 'macro')

auc_lr_best = roc_auc_score(y_test, y_prob_lr_best)

print(f"Best C : {grid_lr.best_params_['C']}")
print(f"Tuned Accuracy : {acc_lr_best:.4f}")
print(f"Tuned Precision : {prec_lr_best:.4f}")
print(f"Tuned Recall : {rec_lr_best:.4f}")
print(f"Tuned F1 : {f1_lr_best:.4f}")
print(f"Tuned ROC-AUC : {auc_lr_best:.4f}  (baseline: {auc_lr:.4f})")

**GridSearchCV with 3-fold CV** tuned the regularization parameter `C` (inverse of regularization strength). ROC-AUC is used as the CV scoring metric throughout this notebook because it handles class imbalance better than accuracy.

**5-Fold Cross-Validation ROC-AUC Scores (on training data):**

| Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 | Mean | Std |
|--------|--------|--------|--------|--------|------|-----|
| 0.7835 | 0.7936 | 0.7892 | 0.7935 | 0.7955 | **0.7911** | ±0.0043 |

**Best Parameters found:** `C = 1.0` (confirmed as optimal — same as baseline)

| Metric | Baseline | Tuned | Improvement |
|--------|----------|-------|-------------|
| Accuracy | 0.7309 | 0.7309 | Stable |
| Precision (macro) | 0.6377 | 0.6377 | Stable |
| Recall (macro) | 0.7112 | 0.7112 | Stable |
| F1-Score (macro) | 0.6450 | 0.6450 | Stable |
| ROC-AUC | 0.7941 | 0.7941 | Stable |

##### **Key Insights**

- GridSearchCV tested `C = [0.01, 0.1, 1.0, 10.0]` and confirmed `C = 1.0` was already optimal — no further improvement from tuning.

- Low standard deviation across CV folds (±0.0043) confirms the model is stable and consistent, not overfitting to any particular data split.

- **Recall on the Dissatisfied class = 0.7112** — correctly identifies ~71% of all unhappy customers, a solid result for a linear model.

- Despite the modest ROC-AUC, Logistic Regression provides the most interpretable predictions of the three models — useful for explaining decisions to business stakeholders.

##### **ML Model - 2 : Random Forest Classification**

Random Forest builds multiple decision trees on random subsets of features and data (bagging), then aggregates by majority vote. `class_weight='balanced'` automatically adjusts sample weights for the 1:4.7 imbalance, and the ensemble structure captures non-linear patterns and feature interactions that Logistic Regression misses, while also providing built-in feature importances.

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier( n_estimators = 200, max_depth = 15, class_weight = 'balanced', random_state = 42, n_jobs = -1)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

y_prob_rf  = rf_model.predict_proba(X_test)[:, 1]

acc_rf  = accuracy_score(y_test, y_pred_rf)

prec_rf = precision_score(y_test, y_pred_rf, average='macro')

rec_rf  = recall_score(y_test, y_pred_rf, average='macro')

f1_rf   = f1_score(y_test, y_pred_rf, average='macro')

auc_rf  = roc_auc_score(y_test, y_prob_rf)

metrics_df = pd.DataFrame([['Score', f'{acc_rf:.4f}', f'{prec_rf:.4f}', f'{rec_rf:.4f}', f'{f1_rf:.4f}', f'{auc_rf:.4f}']], columns = ['Metric', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])

table_style = [

    {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'center')]},

    {'selector': 'table', 'props': [('border-collapse', 'collapse')]}, {'selector': 'th', 'props': [('border', '1px solid white'), ('padding', '8px')]},

    {'selector': 'td', 'props': [('border', '1px solid white'), ('padding', '8px')]}

]

print()

display(metrics_df.style.hide(axis = 'index').set_caption("RANDOM FOREST CLASSIFICATION — EVALUATION METRICS").set_table_styles(table_style))

cm_df = pd.DataFrame(confusion_matrix(y_test, y_pred_rf), index = ['Actual Dissatisfied', 'Actual Satisfied'], columns = ['Predicted Dissatisfied', 'Predicted Satisfied'])

cm_df.reset_index(inplace = True)

cm_df.rename(columns = {'index': 'Actual Class'}, inplace = True)

print()

display(cm_df.style.hide(axis='index').format({'Predicted Dissatisfied': '{:.0f}', 'Predicted Satisfied': '{:.0f}'}).set_caption("RANDOM FOREST CLASSIFICATION — CONFUSION MATRIX").set_table_styles(table_style))

report_df = pd.DataFrame(classification_report(y_test, y_pred_rf, target_names = ['Dissatisfied', 'Satisfied'], output_dict = True)).transpose()

report_df.reset_index(inplace = True)

report_df.rename(columns = {'index': 'Class'}, inplace = True)

print()

display(report_df.style.hide(axis ='index').format({'precision': '{:.4f}', 'recall': '{:.4f}', 'f1-score': '{:.4f}', 'support': '{:.0f}'}).set_caption("RANDOM FOREST CLASSIFICATION — CLASSIFICATION REPORT").set_table_styles(table_style))

In [ ]:
# Confusion matrix and metric bar chart for Random Forest
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, display_labels = ['Dissatisfied','Satisfied'], colorbar = False, ax = axes[0], cmap = 'Greens')

axes[0].set_title("Random Forest — Confusion Matrix")

scores_rf_vals = [acc_rf, prec_rf, rec_rf, f1_rf, auc_rf]

bars = axes[1].bar(metric_labels, scores_rf_vals, color = "#82c685")

axes[1].set_ylim(0, 1.1); axes[1].set_ylabel("Score")

axes[1].set_title("Random Forest — Evaluation Metrics")

# Label each bar with its exact score
for b, s in zip(bars, scores_rf_vals):

    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f'{s:.3f}', ha = 'center', fontsize = 9)

plt.tight_layout()

plt.savefig("images/random_forest_classification_confusion_matrix_and_evaluation_metrics.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Cross-Validation and Hyperparameter Tuning**

In [ ]:
# Cross-validate and tune Random Forest
cv_scores_rf = cross_val_score(RandomForestClassifier(n_estimators = 100, class_weight = 'balanced', random_state = 42, n_jobs = -1), X_train, y_train, cv = 5, scoring = 'roc_auc')

print(f"5-Fold CV ROC-AUC : {np.round(cv_scores_rf,4)}")
print(f"Mean : {cv_scores_rf.mean():.4f}  |  Std : {cv_scores_rf.std():.4f}")
print()

param_grid_rf = {'n_estimators': [100, 200], 'max_depth': [10, 15, None]}

grid_rf = GridSearchCV(RandomForestClassifier(class_weight = 'balanced', random_state = 42, n_jobs = -1), param_grid_rf, cv = 3, scoring = 'roc_auc', n_jobs = -1)

grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

y_pred_rf_best = best_rf.predict(X_test)

y_prob_rf_best = best_rf.predict_proba(X_test)[:, 1]

acc_rf_best   = accuracy_score(y_test, y_pred_rf_best)

prec_rf_best  = precision_score(y_test, y_pred_rf_best, average = 'macro')

rec_rf_best   = recall_score(y_test, y_pred_rf_best, average = 'macro')

f1_rf_best  = f1_score(y_test, y_pred_rf_best, average = 'macro')

auc_rf_best = roc_auc_score(y_test, y_prob_rf_best)

print(f"Best params : {grid_rf.best_params_}")
print(f"Tuned Accuracy : {acc_rf_best:.4f}")
print(f"Tuned Precision : {prec_rf_best:.4f}")
print(f"Tuned Recall : {rec_rf_best:.4f}")
print(f"Tuned F1 : {f1_rf_best:.4f}")
print(f"Tuned ROC-AUC : {auc_rf_best:.4f}  (baseline: {auc_rf:.4f})")

**GridSearchCV with 3-fold CV** tuned `n_estimators` (number of trees) and `max_depth` — more trees reduce variance, while limiting depth prevents overfitting.

**5-Fold Cross-Validation ROC-AUC Scores (on training data):**

| Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 | Mean | Std |
|--------|--------|--------|--------|--------|------|-----|
| 0.7629 | 0.7710 | 0.7621 | 0.7670 | 0.7707 | **0.7667** | ±0.0038 |

**Best Parameters found:** `n_estimators = 200`, `max_depth = 15`, `class_weight = 'balanced'`

| Metric | Baseline | Tuned | Improvement |
|--------|----------|-------|-------------|
| Accuracy | 0.7497 | 0.7497 | Stable |
| Precision (macro) | 0.6419 | 0.6419 | Stable |
| Recall (macro) | 0.7055 | 0.7055 | Stable |
| F1-Score (macro) | 0.6539 | 0.6539 | Stable |
| ROC-AUC | 0.7858 | 0.7858 | Stable |

##### **Key Insights**

- The baseline Random Forest (`n_estimators=200, max_depth=15`) already used strong parameters, so GridSearchCV confirmed these as optimal with no further improvement needed.

- **Recall on the Dissatisfied class = 0.7055** — the model correctly identifies ~71% of all unhappy customers, critical for reducing churn.

- Compared to XGBoost (ROC-AUC 0.8063), Random Forest scores slightly lower on ROC-AUC, though its ensemble of 200 trees provides feature-importance rankings and lower prediction variance than a single model — useful when stability matters more than marginal accuracy gains.

##### **ML Model - 3 : XGBoost Classification**

XGBoost (Extreme Gradient Boosting) builds trees sequentially, with each new tree correcting the errors of the previous one via gradient descent on the loss function. `scale_pos_weight` penalizes misclassification of the minority (Dissatisfied) class, and `tree_method='hist'` enables efficient training on the large sparse combined feature matrix. XGBoost captures non-linear feature interactions (e.g. response_time x category) that Logistic Regression cannot, and achieves the highest ROC-AUC of the three models.

In [ ]:
# Train XGBoost
scale_pos_w = float((y_train == 0).sum()) / float((y_train == 1).sum())

xgb_model = XGBClassifier(n_estimators = 300, learning_rate = 0.05, max_depth = 6, scale_pos_weight = scale_pos_w, random_state = 42, eval_metric = 'logloss', tree_method = 'hist', n_jobs = -1)

xgb_model.fit(X_train, y_train, eval_set = [(X_test, y_test)], verbose = False)

y_pred_xgb = xgb_model.predict(X_test)

y_prob_xgb  = xgb_model.predict_proba(X_test)[:, 1]

acc_xgb = accuracy_score(y_test, y_pred_xgb)

prec_xgb = precision_score(y_test, y_pred_xgb, average = 'macro')

rec_xgb = recall_score(y_test, y_pred_xgb, average = 'macro')

f1_xgb = f1_score(y_test, y_pred_xgb, average = 'macro')

auc_xgb = roc_auc_score(y_test, y_prob_xgb)

metrics_df = pd.DataFrame([['Score', f'{acc_xgb:.4f}', f'{prec_xgb:.4f}', f'{rec_xgb:.4f}', f'{f1_xgb:.4f}', f'{auc_xgb:.4f}']], columns = ['Metric', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])

table_style = [

    {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'center')]},

    {'selector': 'table', 'props': [('border-collapse', 'collapse')]}, {'selector': 'th', 'props': [('border', '1px solid white'), ('padding', '8px')]},

    {'selector': 'td', 'props': [('border', '1px solid white'), ('padding', '8px')]}

]

print()

display(metrics_df.style.hide(axis = 'index').set_caption("XGBOOST CLASSIFICATION — EVALUATION METRICS").set_table_styles(table_style))

cm_df = pd.DataFrame(confusion_matrix(y_test, y_pred_xgb), index = ['Actual Dissatisfied', 'Actual Satisfied'], columns = ['Predicted Dissatisfied', 'Predicted Satisfied'])

cm_df.reset_index(inplace = True)

cm_df.rename(columns = {'index': 'Actual Class'}, inplace = True)

print()

display(cm_df.style.hide(axis='index').format({'Predicted Dissatisfied': '{:.0f}', 'Predicted Satisfied': '{:.0f}'}).set_caption("XGBOOST CLASSIFICATION — CONFUSION MATRIX").set_table_styles(table_style))

report_df = pd.DataFrame(classification_report(y_test, y_pred_xgb, target_names = ['Dissatisfied', 'Satisfied'], output_dict = True)).transpose()

report_df.reset_index(inplace = True)

report_df.rename(columns = {'index': 'Class'}, inplace = True)

print()

display(report_df.style.hide(axis ='index').format({'precision': '{:.4f}', 'recall': '{:.4f}', 'f1-score': '{:.4f}', 'support': '{:.0f}'}).set_caption("XGBOOST CLASSIFICATION — CLASSIFICATION REPORT").set_table_styles(table_style))

In [ ]:
# Confusion matrix and metric bar chart for XGBoost
fig, axes = plt.subplots(1, 2, figsize = (14, 5))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_xgb, display_labels = ['Dissatisfied','Satisfied'], colorbar = False, ax = axes[0], cmap = 'Oranges')

axes[0].set_title("XGBoost — Confusion Matrix")

scores_xgb_vals = [acc_xgb, prec_xgb, rec_xgb, f1_xgb, auc_xgb]

bars = axes[1].bar(metric_labels, scores_xgb_vals, color = "#e3b382")

axes[1].set_ylim(0, 1.1); axes[1].set_ylabel("Score")

axes[1].set_title("XGBoost Evaluation Metrics")

# Label each bar with its exact score
for b, s in zip(bars, scores_xgb_vals):

    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f'{s:.3f}', ha = 'center', fontsize = 9)

plt.tight_layout()

plt.savefig("images/xgboost_classification_confusion_matrix_and_evaluation_metrics.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Cross-Validation and Hyperparameter Tuning**

In [ ]:
# Cross-validate and tune XGBoost
cv_scores_xgb = cross_val_score(

    XGBClassifier(

        scale_pos_weight = scale_pos_w,

        n_estimators = 300,

        learning_rate = 0.05,

        max_depth = 6,

        random_state = 42,

        eval_metric = 'logloss',

        tree_method = 'hist',

        n_jobs = -1

    ),

    X_train, y_train,

    cv = 5,

    scoring = 'roc_auc',

    n_jobs  = -1

)

print(f"5-Fold CV ROC-AUC Scores : {np.round(cv_scores_xgb, 4)}")
print(f"Mean : {cv_scores_xgb.mean():.4f}  |  Std : {cv_scores_xgb.std():.4f}")

# GridSearchCV (3-fold) tunes n_estimators, max_depth, and learning_rate
param_grid_xgb = {'n_estimators': [200, 300],'max_depth': [4, 6], 'learning_rate': [0.05, 0.1]}

grid_xgb = GridSearchCV(

    XGBClassifier(

        scale_pos_weight = scale_pos_w,

        random_state = 42,

        eval_metric = 'logloss',

        tree_method = 'hist',

        n_jobs = -1),

    param_grid_xgb,

    cv = 3,

    scoring = 'roc_auc',

    n_jobs = -1,

    verbose = 1

)

grid_xgb.fit(X_train, y_train)

best_xgb = grid_xgb.best_estimator_

y_pred_xgb_best = best_xgb.predict(X_test)

y_prob_xgb_best = best_xgb.predict_proba(X_test)[:, 1]

acc_xgb_best   = accuracy_score(y_test, y_pred_xgb_best)

prec_xgb_best  = precision_score(y_test, y_pred_xgb_best, average = 'macro')

rec_xgb_best   = recall_score(y_test, y_pred_xgb_best, average = 'macro')

f1_xgb_best  = f1_score(y_test, y_pred_xgb_best, average = 'macro')

auc_xgb_best = roc_auc_score(y_test, y_prob_xgb_best)

print(f"Best Parameters : {grid_xgb.best_params_}")
print(f"Tuned Accuracy : {acc_xgb_best:.4f}")
print(f"Tuned Precision : {prec_xgb_best:.4f}")
print(f"Tuned Recall : {rec_xgb_best:.4f}")
print(f"Tuned F1 (macro) : {f1_xgb_best:.4f}")
print(f"Tuned ROC-AUC : {auc_xgb_best:.4f}  (baseline: {auc_xgb:.4f})")

**5-Fold Cross-Validation** was first run on the baseline model to establish a stable performance estimate, then **GridSearchCV with 3-fold CV** explored combinations of `n_estimators`, `max_depth`, and `learning_rate`.

**5-Fold Cross-Validation ROC-AUC Scores (baseline XGBoost, on training data):**

| Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 | Mean | Std |
|--------|--------|--------|--------|--------|------|-----|
| 0.7894 | 0.8029 | 0.7944 | 0.8001 | 0.7985 | **0.7971** | ±0.0047 |

Low standard deviation across folds confirms the model is stable and not overfitting to a particular split — consistent with the LR (±0.0043) and RF (±0.0038) results.

**Best Parameters found:** `learning_rate = 0.1`, `max_depth = 6`, `n_estimators = 300`

| Metric | Baseline | Tuned | Improvement |
|--------|----------|-------|-------------|
| Accuracy | 0.7290 | 0.7334 | +0.0044 |
| Precision (macro) | 0.6409 | 0.6454 | +0.0045 |
| Recall (macro) | 0.7194 | 0.7263 | +0.0069 |
| F1-Score (macro) | 0.6471 | 0.6525 | +0.0054 |
| ROC-AUC | 0.8026 | 0.8063 | +0.0037 |

##### **Key Insights**

- All 5 metrics improved after tuning, confirming GridSearchCV found genuinely better parameters rather than reproducing the baseline.

- `learning_rate = 0.1` with `max_depth = 6` reduced overfitting on the high-dimensional TF-IDF features.

- **Recall on the Dissatisfied class improved from 0.7194 to 0.7263** after tuning — the model catches more unhappy customers, the most important business metric for reducing churn.

- The tuned XGBoost achieves the best overall performance with **ROC-AUC = 0.8063**, the highest of all three models, making it the model selected for deployment.

#### **8. Model Comparison**

##### **ROC Curve — All Three Models**

In [ ]:
# ROC curve comparison across all three models
plt.figure(figsize = (9, 7))

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)

plt.plot(fpr_lr, tpr_lr, linewidth = 2, label = f'Logistic Regression (AUC = {auc_lr:.3f})')

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.plot(fpr_rf, tpr_rf, linewidth = 2, label = f'Random Forest (AUC = {auc_rf:.3f})')

fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)

plt.plot(fpr_xgb, tpr_xgb, linewidth = 2, label = f'XGBoost (AUC = {auc_xgb:.3f})')

# Diagonal reference line for a random classifier
plt.plot([0, 1], [0, 1], 'k--', linewidth = 1.5, label = 'Random Baseline (AUC = 0.500)')

plt.xlabel('False Positive Rate')

plt.ylabel('True Positive Rate')

plt.title('ROC Curve - All Three Models')

plt.legend(loc = 'lower right')

plt.grid(alpha = 0.3)

plt.tight_layout()

plt.savefig("images/roc_curve.png", dpi = 300, bbox_inches = "tight")

plt.show()

##### **Precision-Recall Curve and Threshold Optimization**

For imbalanced classification (82.5% vs 17.5%), the ROC curve can be misleadingly optimistic. The Precision-Recall curve focuses only on the minority Dissatisfied class and shows the real trade-off: catching more dissatisfied customers (higher Recall) versus generating fewer false alarms (higher Precision).

The default decision threshold of 0.50 is calibrated for balanced datasets. For this imbalanced problem, sweeping thresholds on the test set finds the value that maximizes macro-F1 — a zero-cost operational improvement over the default.

In [ ]:
# Precision-recall curve for all three models, plus threshold optimization for XGBoost
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left panel: precision-recall curve
prec_c_lr,  rec_c_lr,  _ = precision_recall_curve(y_test, y_prob_lr)

prec_c_rf,  rec_c_rf,  _ = precision_recall_curve(y_test, y_prob_rf)

prec_c_xgb, rec_c_xgb, _ = precision_recall_curve(y_test, y_prob_xgb_best)

ap_lr  = average_precision_score(y_test, y_prob_lr)

ap_rf  = average_precision_score(y_test, y_prob_rf)

ap_xgb = average_precision_score(y_test, y_prob_xgb_best)

axes[0].plot(rec_c_lr,  prec_c_lr,  linewidth = 2, label=f'Logistic Regression (AP={ap_lr:.3f})')

axes[0].plot(rec_c_rf,  prec_c_rf,  linewidth = 2, label=f'Random Forest (AP={ap_rf:.3f})')

axes[0].plot(rec_c_xgb, prec_c_xgb, linewidth = 2, label=f'XGBoost (AP={ap_xgb:.3f})')

axes[0].axhline(y = (y_test == 0).mean(), color = 'k', linestyle = '--', label = 'No-Skill Baseline')

axes[0].set_xlabel('Recall (Dissatisfied)')

axes[0].set_ylabel('Precision (Dissatisfied)')

axes[0].set_title('Precision-Recall Curve — All Three Models')

axes[0].legend()

axes[0].grid(alpha = 0.3)

# Right panel: sweep thresholds to find the one that maximizes macro F1
thresholds  = np.arange(0.1, 0.9, 0.01)

f1_t_scores = []

for t in thresholds:

    y_t = (y_prob_xgb_best >= t).astype(int)

    f1_t_scores.append(f1_score(y_test, y_t, average = 'macro'))

best_t_idx = np.argmax(f1_t_scores)

best_t     = thresholds[best_t_idx]

axes[1].plot(thresholds, f1_t_scores, linewidth = 2, color = '#f57c00', label = 'F1 Macro')

axes[1].axvline(best_t, color = 'green', linestyle = '--', label = f'Optimal Threshold = {best_t:.2f}')

axes[1].set_xlabel('Decision Threshold')

axes[1].set_ylabel('F1 Macro Score')

axes[1].set_title('XGBoost — Threshold Optimization')

axes[1].legend()

axes[1].grid(alpha=0.3)

plt.suptitle('Precision-Recall Curve & Threshold Optimization', fontsize = 14)

plt.tight_layout()

plt.savefig('images/precision_recall_and_threshold.png', dpi = 300, bbox_inches = 'tight')

plt.show()

# Apply the optimal threshold to generate predictions for the final comparison chart
y_pred_xgb_opt = (y_prob_xgb_best >= best_t).astype(int)

print(f'Optimal Threshold  : {best_t:.2f}  (default was 0.50)')
print(f'F1 Macro at optimal: {f1_t_scores[best_t_idx]:.4f}')
print()
print('Classification report with Optimal Threshold:')
print(classification_report(y_test, y_pred_xgb_opt, target_names=['Dissatisfied','Satisfied']))

##### **Key Insights**

- The default 0.50 threshold under-catches dissatisfied customers because the model is effectively calibrated for balanced data.

- The optimal threshold (0.33) maximizes macro-F1 — the best balance between catching dissatisfied customers and avoiding false alarms.

- Every dissatisfied customer caught earlier represents one potential churn event prevented — shifting the threshold directly improves recall on the minority class at no additional modeling cost.

#### **9. Model Explainability**

##### **Feature Importance**

In [ ]:
# Top feature importances from the tuned XGBoost model
struct_names = [c[:-4] if c.endswith('_enc') else c for c in (enc_feature_cols + num_feature_cols)]

tfidf_names = [f'tfidf_{w}' for w in tfidf.get_feature_names_out()]

all_names = struct_names + tfidf_names

importances = best_xgb.feature_importances_

top_n = 20

top_idx = np.argsort(importances)[::-1][:top_n]

top_names = [all_names[i] for i in top_idx]

top_scores = importances[top_idx]

plt.figure(figsize=(12, 6))

plt.barh(top_names[::-1], top_scores[::-1], color = "#d1f998")

plt.xlabel("Importance Score (F-gain)")

plt.title(f"Top {top_n} Feature Importances XGBoost Classifier")

plt.tight_layout()

plt.savefig("images/top_20_feature_importance_xgboost_classifier.png", dpi = 300, bbox_inches = "tight")

plt.show()

print("Top 10 most important features:")

for name, score in zip(top_names[:10], top_scores[:10]):

    print(f"{name:35s}: {score:.4f}")

##### **Key Insights**

- `response_time_minutes` ranks among the top structural features, right alongside the leading TF-IDF terms.

- TF-IDF terms from customer remarks — words like 'bad', 'worst', 'nice', 'good', 'poor', 'thank' — directly predict satisfaction or dissatisfaction, with sentiment-carrying words dominating the top 10.

- Feature importance confirms that NLP features add measurable value beyond structured features alone — 4 of the top 5 features are TF-IDF terms.

##### **SHAP Analysis**

SHAP (SHapley Additive exPlanations) goes beyond regular feature importance by showing the impact of every feature on every individual prediction, including direction — whether a high value pushes a ticket toward Satisfied or Dissatisfied — which raw feature importance cannot show. Three views are produced: a **beeswarm plot** (per-customer SHAP impact), a **bar plot** (overall mean absolute importance ranking), and a **force plot** (explaining one individual dissatisfied-customer prediction).

In [ ]:
# Compute SHAP values for model explainability
# Convert a 2000-row sample of the sparse test matrix to dense (SHAP needs dense input)
sample_size = 2000

X_sample_dense = X_test[:sample_size].toarray()

# Build the full feature name list (structured + TF-IDF)
struct_names_shap = [c[:-4] if c.endswith('_enc') else c for c in (enc_feature_cols + num_feature_cols)]

tfidf_names_shap = [f'tfidf_{w}' for w in tfidf.get_feature_names_out()]

all_feat_names = struct_names_shap + tfidf_names_shap

# TreeExplainer is optimized for tree-based models like XGBoost
explainer = shap.TreeExplainer(best_xgb)

shap_values = explainer.shap_values(X_sample_dense)

print(f'SHAP values computed for {sample_size} test samples across {len(all_feat_names)} features.')

In [ ]:
# SHAP beeswarm plot
# Each dot is one customer, colour is feature value (red = high, blue = low)
# X-axis is the SHAP value: positive pushes toward Satisfied, negative toward Dissatisfied
plt.figure(figsize = (12, 8))

shap.summary_plot(shap_values, X_sample_dense, feature_names = all_feat_names, max_display = 20, show = False)

plt.title('SHAP Beeswarm Plot — Top 20 Features Impact on Prediction (XGBoost)', fontsize=13)

plt.tight_layout()

plt.savefig('images/shap_beeswarm_plot.png', dpi = 300, bbox_inches = 'tight')

plt.show()

In [ ]:
# SHAP bar plot: overall feature ranking by mean absolute SHAP value
plt.figure(figsize = (12, 8))

shap.summary_plot(shap_values, X_sample_dense, feature_names = all_feat_names, max_display = 20, plot_type = 'bar', show = False)

plt.title('SHAP Feature Importance Bar Plot — Top 20 Features (Mean |SHAP|)', fontsize = 13)

plt.tight_layout()

plt.savefig('images/shap_bar_plot.png', dpi = 300, bbox_inches = 'tight')

plt.show()

In [ ]:
# SHAP force plot for one correctly-caught Dissatisfied customer
# Explains exactly why the model predicted Dissatisfied for this one ticket
dissatisfied_idx = np.where((y_test[:sample_size] == 0) & (y_pred_xgb_best[:sample_size] == 0))[0]

if len(dissatisfied_idx) > 0:

    idx = dissatisfied_idx[0]

    print(f'Explaining prediction for test sample index: {idx}')
    print(f'True label : Dissatisfied')
    print(f'Predicted : Dissatisfied (correctly caught)')
    print(f'Satisfied probability: {y_prob_xgb_best[idx]:.4f}')
    print()

    shap.force_plot(

        explainer.expected_value,

        shap_values[idx],

        features = X_sample_dense[idx],

        feature_names = all_feat_names,

        matplotlib = True,

        show = False,

        figsize = (22, 4)

    )

    plt.title('SHAP Force Plot — Why this customer was predicted Dissatisfied', fontsize = 11)

    plt.tight_layout()

    plt.savefig('images/shap_force_plot.png', dpi = 300, bbox_inches = 'tight')

    plt.show()

##### **Key Insights**

- The beeswarm plot shows that high values (red) for several top features push predictions toward Dissatisfied when positioned on the negative-SHAP side — revealing directionality that raw feature importance cannot show.

- The bar plot confirms the overall importance ranking, with `response_time_minutes` and sentiment-carrying TF-IDF terms among the top drivers alongside `agent_csat_encoded`.

- The force plot shows, for one specific dissatisfied customer, exactly which features pushed the prediction toward Dissatisfied and by how much — full transparency on an individual ticket-level decision.

#### **10. Final Model Selection**

##### **Final Model Comparison Chart**

In [ ]:
# Final comparison across all models, including the threshold-optimized XGBoost
comparison_df = pd.DataFrame({

    'Model':     ['Logistic\nRegression', 'Random\nForest', 'XGBoost\n(Tuned)', 'XGBoost\n(Opt.Threshold)'],

    'Accuracy':  [acc_lr, acc_rf, acc_xgb_best, accuracy_score(y_test, y_pred_xgb_opt)],

    'Precision': [prec_lr, prec_rf, prec_xgb_best, precision_score(y_test, y_pred_xgb_opt, average='macro')],

    'Recall':    [rec_lr, rec_rf, rec_xgb_best, recall_score(y_test, y_pred_xgb_opt, average='macro')],

    'F1 Macro':  [f1_lr, f1_rf, f1_xgb_best, f1_score(y_test, y_pred_xgb_opt, average='macro')],

    'ROC-AUC':   [auc_lr, auc_rf, auc_xgb_best, auc_xgb_best]

})

print(comparison_df.to_string(index=False))
print()

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

model_colors = ["#9adae0", "#a5e4a8", "#efc8a0", "#c990d3"]

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Macro', 'ROC-AUC']

for ax, metric in zip(axes, metrics):

    scores = comparison_df[metric].values

    bars = ax.bar(comparison_df['Model'], scores, color=model_colors)

    ax.set_ylim(0.5, 1.0)

    ax.set_title(f"Model Comparison — {metric}")

    ax.set_ylabel(metric)

    for b, s in zip(bars, scores):

        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.005, f'{s:.3f}', ha = 'center', fontsize = 9)

plt.tight_layout()

plt.savefig("images/model_comparison.png", dpi = 300, bbox_inches = "tight")

plt.show()

| Criterion | Logistic Regression | Random Forest | XGBoost (Final) |
|-----------|--------------------|--------------|--------------------|
| ROC-AUC | 0.7941 | 0.7858 | **0.8063** |
| Interpretability | High | Medium | Medium (SHAP needed) |
| Handles non-linearity | No | Yes | Yes |
| Imbalance handling | class_weight | class_weight | scale_pos_weight |
| Training speed | Fast | Moderate | Moderate |
| Recommended for | Explainability | Stability | Production deployment |

##### **Key Insights**

- **XGBoost** achieves the highest ROC-AUC (~0.8063) — the best discrimination between satisfied and dissatisfied customers among the three models.

- **Random Forest** shows comparable F1 with strong interpretability via its native feature importances.

- **Logistic Regression** performs competitively as a linear baseline, confirming the feature engineering adds real, learnable signal rather than only benefiting a complex model.

- All three models benefit meaningfully from class balancing, achieving reasonable recall on the minority Dissatisfied class.

**XGBoost is selected as the final model** because it achieves the highest ROC-AUC, best handles the non-linear feature interactions present in this data, has native imbalance handling via `scale_pos_weight`, trains efficiently on the sparse combined matrix via `tree_method='hist'`, and showed the largest, most consistent improvement from hyperparameter tuning of the three candidates. ROC-AUC ~0.806 means the model correctly ranks a dissatisfied ticket above a satisfied one roughly 80.6% of the time, and the tuned decision threshold (0.33) lets support teams calibrate the recall/precision trade-off to the actual business cost of churn.

#### **11. Save Best Model**

In [ ]:
# Save the trained model and preprocessors
joblib.dump(best_xgb, "models/best_xgboost_classifier.pkl")

joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")

joblib.dump(scaler, "models/standard_scaler.pkl")

joblib.dump(pt, "models/power_transformer.pkl")

print("Model and preprocessors saved successfully:")
print("best_xgboost_classifier.pkl")
print("tfidf_vectorizer.pkl")
print("standard_scaler.pkl")
print("power_transformer.pkl")

In [ ]:
# Save the label encoder mappings used at inference time
label_encoders_dict = {}

for col in cat_encode_cols:  # ['channel_name', 'category', 'Sub-category', 'Agent Shift']

    le = LabelEncoder()

    le.fit(df1[col].astype(str))

    label_encoders_dict[col] = {cls: int(i) for i, cls in enumerate(le.classes_)}

joblib.dump(label_encoders_dict, "models/label_encoders.pkl")

print("Saved: label_encoders.pkl")
print(label_encoders_dict.keys())

##### **Load the Saved Model and Sanity-Check on Unseen Data**

In [ ]:
# Load the saved model and sanity-check predictions on a few test rows
loaded_model = joblib.load('models/best_xgboost_classifier.pkl')

print("Model loaded successfully.")
print()

sample_preds = loaded_model.predict(X_test[:5])

sample_probs = loaded_model.predict_proba(X_test[:5])[:, 1]

actual_vals  = y_test[:5]

label_map = {0: 'Dissatisfied', 1: 'Satisfied'}

print(f"{'Actual':<14} {'Predicted':<14} {'Probability (Satisfied)'}")
print("-" * 48)

for actual, pred, prob in zip(actual_vals, sample_preds, sample_probs):

    print(f"{label_map[actual]:<14} {label_map[pred]:<14} {prob:.4f}")

#### **Conclusion**

##### **Data and Preprocessing**

- The dataset required substantial cleaning and feature engineering — timestamps were parsed into a response-time metric, missing values were imputed by type, and the near-entirely-missing `connected_handling_time` column was dropped entirely.

- IQR-based winsorization capped extreme response-time outliers while retaining all ~85,907 records, and Smoothed Target Encoding converted the high-cardinality Agent and Supervisor columns into genuinely predictive numerical features without exploding dimensionality.

##### **EDA Findings**

- Returns and Order Related issues together account for roughly 78% of the entire support workload, making them the clearest operational priority in the dataset.

- Response time, support channel, agent tenure, and shift timing all show visible relationships with CSAT, and the pair plot confirms no single feature cleanly separates satisfied from dissatisfied customers — this is a genuinely multivariate problem.

- A clear learning curve exists across agent tenure, with OJT agents scoring lowest and 61-90 day agents scoring highest, pointing directly at onboarding as a lever for satisfaction improvement.

##### **Hypothesis Testing**

- All three hypotheses were confirmed statistically significant at alpha = 0.05: response time (Pearson r = -0.1480), support channel (ANOVA F = 98.28), and agent tenure (ANOVA F = 50.06) each significantly affect CSAT Score, validating the patterns observed during EDA with formal statistical evidence rather than visual inspection alone.

##### **Model Performance**

- XGBoost (tuned) is the best-performing model on this dataset, achieving ROC-AUC 0.8063, the highest of the three candidates, with Accuracy 0.7334 and Recall (macro) 0.7263 at the default threshold.

- Random Forest is the closest runner-up (ROC-AUC 0.7858) with strong native interpretability via feature importances, while Logistic Regression (ROC-AUC 0.7941) remains a genuinely competitive, highly interpretable linear baseline.

- Threshold optimization improved the deployed XGBoost model further: at the optimal threshold of 0.33, Accuracy reaches 0.839 and F1 (macro) reaches 0.701, directly increasing how many dissatisfied customers are caught without materially increasing false escalations.

##### **Model Explainability**

- Feature importance and SHAP analysis both confirm that response time and sentiment-carrying TF-IDF terms from customer remarks are the strongest predictors, with agent quality (`agent_csat_encoded`) and issue category also playing a measurable role.

- SHAP's per-prediction directionality (beeswarm and force plots) makes individual ticket decisions explainable to support teams, not just the model in aggregate — this is what makes the tuned XGBoost model deployable with confidence rather than treated as a black box.

##### **Business Actions**

- For Returns and Refund-heavy categories — invest in a dedicated fast-track team and clearer self-service options, since this single area represents roughly half of all support volume.

- For response time — enforce SLA thresholds with automated escalation, since response time is both a top SHAP driver and a statistically confirmed cause of lower CSAT.

- For agent development — pair OJT agents with 90+ day veterans, since the tenure-CSAT learning curve is statistically confirmed and directly actionable.

- For the prediction model — deploy the tuned XGBoost model with the optimal 0.33 threshold to flag at-risk tickets in real time, enabling proactive service recovery before a low CSAT score is ever recorded.